# G1 — EVA-02 + IA³ / AdaLoRA Smoke Test

**Amaç:** Veri setine dokunmadan, rastgele tensörlerle iki PEFT yönteminin de kurulduğunu, forward/backward yaptığını, parametre sayılarının doğru olduğunu ve adapter save/load'un çalıştığını kanıtlamak.

**Kullanım:** Runtime → Change runtime type → **GPU** seçin. Repo URL'sini aşağıdaki hücreye girin. Colab free oturumları kopabilir; JSON ve checkpoint'ler Drive'a yazılır.

In [ ]:
!nvidia-smi
import torch, timm, peft, sys
print('Python:', sys.version.split()[0])
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('timm:', timm.__version__, '| peft:', peft.__version__)

In [ ]:
# Colab'da torch/torchvision hazır; timm/peft sürümlerini sabitle.
!pip -q install "timm==1.0.31" "peft==0.21.3"

In [ ]:
import os

REPO_URL = ""  # <-- GitHub repo URL'ini buraya yaz
PROJECT_DIR = "/content/derin-ogrenme-proje"

if REPO_URL and not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
elif not os.path.exists(PROJECT_DIR):
    raise SystemExit("Proje klasörü yok. REPO_URL doldur veya klasörü Drive'dan kopyala.")

%cd {PROJECT_DIR}
!ls

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/derin_ogrenme_proje/results'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output:', OUTPUT_DIR)

In [ ]:
# Hedef modül isimlerini doğrula (pretrained indirmez).
!python scripts/inspect_modules.py

In [ ]:
# G1 smoke test: iki yöntem, 20'şer dummy adım.
# İlk seferde indirme + test 5-10 dk sürebilir.
!python scripts/smoke_test.py --method both --steps 20 --output-dir {OUTPUT_DIR}

In [ ]:
import json
from pathlib import Path

path = Path(OUTPUT_DIR) / 'g1_smoke_test.json'
print(path.read_text(encoding='utf-8'))

## G1 kabul kriteri

- [ ] IA³ ve AdaLoRA modelleri kuruldu.
- [ ] Çıkış şekli `[2, 12]`, loss sonlu.
- [ ] `head` eğitilebilir; parametre oranı kayıtlı.
- [ ] AdaLoRA `rank_layers_changed > 0`.
- [ ] `roundtrip_ok: true`.
- [ ] JSON Drive'a kaydedildi.

Sonuçları `results/g1_smoke_test.json` olarak commit edin; checkpoint'ler git'e girmesin.